# Processing Book of Mormon text

## Full Text to Folder

In [12]:
# Important file paths
BOM_PATH = r'C:\Users\evere\Documents\rel_proj\bom_text\content\bom_full_text.html'
CONTENT_PATH = r'C:\Users\evere\Documents\rel_proj\bom_text\content\bom_content.html'

### Extract Relevant Text

First, we will extract just the relevant .html, including only the embedded styles and the actual text of the book. This will exclude all boilerplate content.

In [13]:
# Extract relevant .html
from bs4 import BeautifulSoup

def load_html_soup(html):
    with open(BOM_PATH, 'r', encoding='utf-8') as html:
        soup = BeautifulSoup(html, 'html.parser')
    return soup

def remove_class(soup, class_):
    for element in soup.find_all(class_=class_):
        element.decompose()

def save_soup(soup, save_path):
    with open(save_path, mode='w', encoding='utf-8') as save:
        save.write(soup.prettify())

soup = load_html_soup(BOM_PATH)
remove_class(soup, 'pg-boilerplate')
save_soup(soup, CONTENT_PATH)

### Organize Text

Now, we need to take the cleaned-up html and convert it into a file structure to use later. We will have a folder for each book, with a single .txt file for each chapter. The chapter content will be one block paragraph.

First, we will load the processed .html as the new soup.

In [14]:
# Load the processed .html as the new soup
soup = load_html_soup(CONTENT_PATH)

Then we will create the root folder for the books and chapters.

In [15]:
# Make the root folder structure
from pathlib import Path

root_path = Path(r'content\the_bom')
root_path.mkdir(exist_ok=True)

Now we will get a reference to each book tag and get the name of each book.

In [16]:
# Each book is contained within a <div class="chapter"> tag
books = soup.select('.chapter')

# We will also get the name for each book to use in our folders
find_id_text = lambda id: soup.select(id)[0].parent.get_text().split('\n')[1].lower().strip().replace(' ', '_')
find_book_ids = lambda : [element.get('href', '') for element in soup.select('.pginternal')]
book_names = [find_id_text(book_id) for book_id in find_book_ids()]
print(book_names)

['the_first_book_of_nephi', 'the_second_book_of_nephi', 'the_book_of_jacob', 'the_book_of_enos', 'the_book_of_jarom', 'the_book_of_omni', 'the_words_of_mormon', 'the_book_of_mosiah', 'the_book_of_alma', 'the_book_of_helaman', 'third_book_of_nephi', 'fourth_nephi', 'the_book_of_mormon', 'the_book_of_ether', 'the_book_of_moroni']


Now, we will define a few helper functions to help process chapters and verses.

In [17]:
# Helper functions
import re

def save_chapter(chapter_start, book_path, file_name=None):
    """
    Parses a chapter and saves saves the text to a file 'chapter_#.txt' in the provided book_path directory.
    """
    if not file_name:
        number = chapter_start.get_text().strip().split()[-1]
        save_file = f'chapter_{int(number)}.txt'
    else:
        save_file = 'chapter_1.txt'
    with open(book_path.joinpath(save_file), 'w', encoding='utf-8') as f:
        f.write(process_chapter(chapter_start))

def process_chapter(chapter_heading):
    """
    Parses the chapter text following the provided heading tag.
    """
    final = ''
    verse = chapter_heading.find_next_sibling()
    while verse.name == 'p':
        final += process_verse(verse.get_text()) + ' '
        verse = verse.find_next_sibling()
        if verse is None:
            break
    return final.strip()

def process_verse(verse_text):
    """
    Strips the leading versification and removes newlines from the provided text.
    """
    processed = re.match(r'\d+:\d+\s((.|\n)*)', ' '.join(verse_text.split()))
    return processed.group(1)

Now we will iterate through each book, create a new folder for each one, and extract and save the text of each chapter as a seperate file.

In [18]:
for i in range(len(books)):
    book = books[i]
    book_name = book_names[i]
    book_path = root_path.joinpath(f'{i+1}_{book_name}')
    book_path.mkdir(exist_ok=True)

    chapter_headings = book.find_all('h3')
    for chapter in chapter_headings:
        if not chapter.get_text().isupper():
            save_chapter(chapter, book_path=book_path)
    if len(chapter_headings) == 0:
        save_chapter(book.find(), book_path=book_path, file_name='chapter_1.txt')

# Conclusion

Running all of the above code should extract and organize the text.